# PRMP vs Standard Aggregation on RelBench rel-stack

**Predictive Residual Message Passing (PRMP)** implements a predict-subtract-propagate mechanism
for cross-table feature prediction on RelBench rel-stack FK links.

This notebook:
1. Loads RelBench FK-link cross-table prediction datasets (9 FK links, ~11 examples each in demo mode)
2. Computes a **2D diagnostic** (cardinality × predictability R²) per FK link
3. Trains **PRMP** (zero-init final layer, LayerNorm residuals, gradient-detached dual-loss) vs baselines (Linear Regression, standard MLP)
4. Performs **regime analysis** showing correlation between cardinality×predictability and PRMP improvement

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install
_pip('loguru==0.7.3')

# Core packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1',
         'scipy==1.15.3', 'matplotlib==3.10.0')
    _pip('torch', '--extra-index-url', 'https://download.pytorch.org/whl/cpu')

In [ ]:
import gc
import json
import math
import os
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from loguru import logger
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy import stats
import matplotlib.pyplot as plt

# Configure logger for notebook
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

# Detect device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-b2d5b0-predictive-residual-message-passing-filt/main/experiment_iter2_prmp_vs_standar/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
n_datasets = len(data.get("datasets", []))
total_examples = sum(len(ds.get("examples", [])) for ds in data.get("datasets", []))
logger.info(f"Loaded {n_datasets} FK-link datasets, {total_examples} total examples")

## Configuration

All tunable hyperparameters. Start with minimum values for fast demo execution.

In [ ]:
# --- Tunable parameters ---
HIDDEN_DIM = 8        # Original: 64
EPOCHS = 10           # Original: 150
LEARNING_RATE = 0.001 # Original: 0.001
WEIGHT_DECAY = 1e-5   # Original: 1e-5
MIN_EXAMPLES = 5      # Minimum examples per dataset to run experiment

## Phase 1: Helper Functions

Parse JSON feature strings from the dataset into numerical arrays.

In [ ]:
def parse_feature_dict(s: str) -> dict:
    """Parse a JSON string of features into a dict."""
    try:
        return json.loads(s)
    except (json.JSONDecodeError, TypeError):
        return {}

## Phase 2: FK Diagnostic (Cardinality × Predictability R²)

For each FK link, compute cross-table predictability R² via cross-validated Linear Regression and extract cardinality metadata.

In [ ]:
def compute_fk_diagnostic(data: dict) -> dict:
    """Compute cross-table predictability R² and cardinality for each FK link."""
    logger.info("Computing FK diagnostic (cardinality x predictability R²)")
    fk_diagnostic = {}

    for ds in data.get("datasets", []):
        ds_name = ds["dataset"]
        examples = ds["examples"]
        if not examples:
            continue

        # Extract metadata from first example
        ex0 = examples[0]
        child_table = ex0.get("metadata_child_table", "unknown")
        parent_table = ex0.get("metadata_parent_table", "unknown")
        fkey_col = ex0.get("metadata_fkey_col", "unknown")
        cardinality_mean = ex0.get("metadata_cardinality_mean", 0.0)
        cardinality_median = ex0.get("metadata_cardinality_median", 0.0)
        coverage = ex0.get("metadata_coverage", 0.0)

        # Build feature matrices
        X_list, Y_list = [], []
        for ex in examples:
            inp = parse_feature_dict(ex["input"])
            out = parse_feature_dict(ex["output"])
            if inp and out:
                X_list.append(list(inp.values()))
                Y_list.append(list(out.values()))

        if len(X_list) < MIN_EXAMPLES:
            logger.info(f"  {ds_name}: too few examples ({len(X_list)}), skipping R²")
            fk_diagnostic[ds_name] = {
                "cardinality_mean": cardinality_mean,
                "cardinality_median": cardinality_median,
                "coverage": coverage,
                "predictability_r2": 0.0,
                "child_table": child_table,
                "parent_table": parent_table,
                "fkey_col": fkey_col,
                "n_examples": len(X_list),
            }
            continue

        X = np.array(X_list, dtype=np.float64)
        Y = np.array(Y_list, dtype=np.float64)

        # Handle NaN/Inf
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        Y = np.nan_to_num(Y, nan=0.0, posinf=0.0, neginf=0.0)

        # Compute R² via cross-validation
        try:
            lr = LinearRegression()
            n_cv = min(5, max(2, len(X) // 10))
            scores = cross_val_score(lr, X, Y, cv=n_cv, scoring="r2")
            r2 = float(np.mean(scores))
        except Exception as e:
            logger.warning(f"  {ds_name}: R² computation failed: {e}")
            r2 = 0.0

        fk_diagnostic[ds_name] = {
            "cardinality_mean": cardinality_mean,
            "cardinality_median": cardinality_median,
            "coverage": coverage,
            "predictability_r2": r2,
            "child_table": child_table,
            "parent_table": parent_table,
            "fkey_col": fkey_col,
            "n_examples": len(X_list),
        }
        logger.info(f"  {ds_name}: card={cardinality_mean:.2f}, R²={r2:.4f}, n={len(X_list)}")

    return fk_diagnostic

fk_diagnostic = compute_fk_diagnostic(data)
logger.info(f"Computed diagnostic for {len(fk_diagnostic)} FK links")

## Phase 3: PRMP and Baseline Model Definitions

**PRMPPredictor**: Uses zero-init final layer, LayerNorm residuals, and gradient-detached dual-loss training (0.7 direct + 0.3 residual refinement).

**BaselineMLP**: Standard 3-layer MLP with no residual mechanism.

In [ ]:
def build_prmp_predictor(in_dim, out_dim, hidden_dim=None):
    """Build a 2-layer prediction MLP with zero-init final layer (PRMP core)."""
    if hidden_dim is None:
        hidden_dim = min(in_dim, out_dim)
    hidden_dim = max(hidden_dim, 4)  # Minimum hidden dim

    pred_mlp = nn.Sequential(
        nn.Linear(in_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, out_dim),
    )
    # Zero-init final layer so initial residuals ≈ raw features
    nn.init.zeros_(pred_mlp[-1].weight)
    nn.init.zeros_(pred_mlp[-1].bias)
    return pred_mlp


class PRMPPredictor:
    """PRMP-style cross-table predictor.

    1. pred_mlp predicts child from parent (zero-init final layer)
    2. During training: residuals (child - predicted) are LayerNorm'd,
       combined with parent via update_lin for a refined prediction.
       Dual loss: direct prediction + residual-based refinement.
    3. At inference: pred_mlp(parent) is the output prediction.
    """

    def __init__(self, device, in_dim, out_dim, hidden_dim=32,
                 lr=0.001, epochs=100):
        self.device = device
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.hidden_dim = hidden_dim
        self.lr = lr
        self.epochs = epochs

        self.pred_mlp = build_prmp_predictor(in_dim, out_dim, hidden_dim).to(device)
        self.residual_norm = nn.LayerNorm(out_dim).to(device)
        self.update_lin = nn.Sequential(
            nn.Linear(in_dim + out_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        ).to(device)

    def train_model(self, X_train, Y_train, X_val=None, Y_val=None):
        X_t = torch.tensor(X_train, dtype=torch.float32, device=self.device)
        Y_t = torch.tensor(Y_train, dtype=torch.float32, device=self.device)

        params = (list(self.pred_mlp.parameters()) +
                  list(self.residual_norm.parameters()) +
                  list(self.update_lin.parameters()))
        optimizer = torch.optim.Adam(params, lr=self.lr, weight_decay=WEIGHT_DECAY)
        loss_fn = nn.MSELoss()

        train_losses = []
        val_losses = []

        for epoch in range(self.epochs):
            self.pred_mlp.train()
            self.residual_norm.train()
            self.update_lin.train()

            predicted = self.pred_mlp(X_t)
            direct_loss = loss_fn(predicted, Y_t)

            residual = Y_t - predicted.detach()
            normed_residual = self.residual_norm(residual)
            combined = torch.cat([X_t, normed_residual], dim=-1)
            refined = self.update_lin(combined)
            refine_loss = loss_fn(refined, Y_t)

            loss = 0.7 * direct_loss + 0.3 * refine_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step()

            train_losses.append(float(direct_loss.item()))

            if X_val is not None and Y_val is not None and (epoch + 1) % 10 == 0:
                val_loss = self._eval_loss(X_val, Y_val)
                val_losses.append({"epoch": epoch + 1, "val_loss": val_loss})

        return {"train_losses": train_losses, "val_losses": val_losses}

    def _eval_loss(self, X, Y):
        self.pred_mlp.eval()
        with torch.no_grad():
            X_t = torch.tensor(X, dtype=torch.float32, device=self.device)
            Y_t = torch.tensor(Y, dtype=torch.float32, device=self.device)
            predicted = self.pred_mlp(X_t)
            loss = nn.MSELoss()(predicted, Y_t)
        return float(loss.item())

    def predict(self, X):
        self.pred_mlp.eval()
        with torch.no_grad():
            X_t = torch.tensor(X, dtype=torch.float32, device=self.device)
            predicted = self.pred_mlp(X_t)
            return predicted.cpu().numpy()

    def predict_with_residual(self, X, Y):
        self.pred_mlp.eval()
        self.residual_norm.eval()
        self.update_lin.eval()
        with torch.no_grad():
            X_t = torch.tensor(X, dtype=torch.float32, device=self.device)
            Y_t = torch.tensor(Y, dtype=torch.float32, device=self.device)
            predicted = self.pred_mlp(X_t)
            residual = Y_t - predicted
            normed_residual = self.residual_norm(residual)
            combined = torch.cat([X_t, normed_residual], dim=-1)
            refined = self.update_lin(combined)
        return {
            "predicted_child": predicted.cpu().numpy(),
            "residual": residual.cpu().numpy(),
            "refined_output": refined.cpu().numpy(),
            "residual_magnitude": float(torch.mean(torch.abs(residual)).item()),
        }

    def get_prediction_r2(self, X, Y):
        self.pred_mlp.eval()
        with torch.no_grad():
            X_t = torch.tensor(X, dtype=torch.float32, device=self.device)
            Y_t = torch.tensor(Y, dtype=torch.float32, device=self.device)
            predicted = self.pred_mlp(X_t)
            ss_res = ((Y_t - predicted) ** 2).sum().item()
            ss_tot = ((Y_t - Y_t.mean(dim=0)) ** 2).sum().item()
            r2 = 1.0 - ss_res / max(ss_tot, 1e-10)
        return r2


class BaselineMLP:
    """Standard MLP baseline (no residual mechanism)."""

    def __init__(self, device, in_dim, out_dim, hidden_dim=32,
                 lr=0.001, epochs=100):
        self.device = device

        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        ).to(device)

        self.lr = lr
        self.epochs = epochs

    def train_model(self, X_train, Y_train, X_val=None, Y_val=None):
        X_t = torch.tensor(X_train, dtype=torch.float32, device=self.device)
        Y_t = torch.tensor(Y_train, dtype=torch.float32, device=self.device)

        optimizer = torch.optim.Adam(self.mlp.parameters(), lr=self.lr, weight_decay=WEIGHT_DECAY)
        loss_fn = nn.MSELoss()

        train_losses = []
        val_losses = []

        for epoch in range(self.epochs):
            self.mlp.train()
            output = self.mlp(X_t)
            loss = loss_fn(output, Y_t)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.mlp.parameters(), 1.0)
            optimizer.step()
            train_losses.append(float(loss.item()))

            if X_val is not None and Y_val is not None and (epoch + 1) % 10 == 0:
                val_loss = self._eval_loss(X_val, Y_val)
                val_losses.append({"epoch": epoch + 1, "val_loss": val_loss})

        return {"train_losses": train_losses, "val_losses": val_losses}

    def _eval_loss(self, X, Y):
        self.mlp.eval()
        with torch.no_grad():
            X_t = torch.tensor(X, dtype=torch.float32, device=self.device)
            Y_t = torch.tensor(Y, dtype=torch.float32, device=self.device)
            output = self.mlp(X_t)
            loss = nn.MSELoss()(output, Y_t)
        return float(loss.item())

    def predict(self, X):
        self.mlp.eval()
        with torch.no_grad():
            X_t = torch.tensor(X, dtype=torch.float32, device=self.device)
            output = self.mlp(X_t)
        return output.cpu().numpy()

    def get_prediction_r2(self, X, Y):
        self.mlp.eval()
        with torch.no_grad():
            X_t = torch.tensor(X, dtype=torch.float32, device=self.device)
            Y_t = torch.tensor(Y, dtype=torch.float32, device=self.device)
            predicted = self.mlp(X_t)
            ss_res = ((Y_t - predicted) ** 2).sum().item()
            ss_tot = ((Y_t - Y_t.mean(dim=0)) ** 2).sum().item()
            r2 = 1.0 - ss_res / max(ss_tot, 1e-10)
        return r2

print("PRMPPredictor and BaselineMLP defined.")

## Phase 4-5: Cross-Table Prediction Experiment

Run PRMP vs baselines (Linear Regression, standard MLP) on all FK link datasets. For each dataset:
- Parse features, split train/test by fold
- Train all three models and compute R², MSE, MAE

In [ ]:
def run_cross_table_experiment(data, device, hidden_dim, epochs, lr):
    """Run PRMP vs baseline cross-table prediction on all FK link datasets."""
    dataset_results = []
    all_metrics = []  # For summary table

    for ds_idx, ds in enumerate(data.get("datasets", [])):
        ds_name = ds["dataset"]
        examples = ds["examples"]
        logger.info(f"[{ds_idx+1}/{len(data['datasets'])}] Processing {ds_name} ({len(examples)} examples)")

        if len(examples) < MIN_EXAMPLES:
            logger.warning(f"  Skipping {ds_name}: too few examples")
            continue

        # Parse features
        X_list, Y_list, metadata_list = [], [], []
        child_feat_names = examples[0].get("metadata_child_feature_names", [])
        parent_feat_names = examples[0].get("metadata_parent_feature_names", [])

        for ex in examples:
            inp = parse_feature_dict(ex["input"])
            out = parse_feature_dict(ex["output"])
            if inp and out:
                X_list.append([float(v) if v is not None else 0.0 for v in inp.values()])
                Y_list.append([float(v) if v is not None else 0.0 for v in out.values()])
                metadata_list.append(ex)

        if len(X_list) < MIN_EXAMPLES:
            logger.warning(f"  Skipping {ds_name}: too few valid examples after parsing")
            continue

        X = np.array(X_list, dtype=np.float64)
        Y = np.array(Y_list, dtype=np.float64)
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        Y = np.nan_to_num(Y, nan=0.0, posinf=0.0, neginf=0.0)

        in_dim = X.shape[1]
        out_dim = Y.shape[1]

        # Split by fold
        folds = np.array([ex.get("metadata_fold", 0) for ex in metadata_list])

        # Use fold 0 as test, rest as train
        test_mask = folds == 0
        train_mask = ~test_mask

        if test_mask.sum() < 2 or train_mask.sum() < 2:
            # Fallback: random 80/20 split
            n = len(X)
            perm = np.random.RandomState(42).permutation(n)
            split = max(2, int(0.8 * n))
            train_mask = np.zeros(n, dtype=bool)
            train_mask[perm[:split]] = True
            test_mask = ~train_mask

        X_train, Y_train = X[train_mask], Y[train_mask]
        X_test, Y_test = X[test_mask], Y[test_mask]

        logger.info(f"  in_dim={in_dim}, out_dim={out_dim}, train={len(X_train)}, test={len(X_test)}")

        # --- Baseline 1: Linear Regression ---
        t0 = time.time()
        try:
            lr_model = LinearRegression()
            lr_model.fit(X_train, Y_train)
            lr_preds_test = lr_model.predict(X_test)
            lr_preds_all = lr_model.predict(X)
            lr_r2 = r2_score(Y_test, lr_preds_test)
            lr_mse = mean_squared_error(Y_test, lr_preds_test)
            lr_mae = mean_absolute_error(Y_test, lr_preds_test)
            lr_time = time.time() - t0
            logger.info(f"  LinearReg: R²={lr_r2:.4f}, MSE={lr_mse:.4f}, time={lr_time:.2f}s")
        except Exception as e:
            logger.warning(f"  LinearReg failed: {e}")
            lr_preds_all = np.zeros_like(Y)
            lr_r2, lr_mse, lr_mae, lr_time = 0.0, 0.0, 0.0, 0.0

        # --- Baseline 2: Standard MLP ---
        t0 = time.time()
        try:
            baseline_mlp = BaselineMLP(
                device, in_dim=in_dim, out_dim=out_dim,
                hidden_dim=hidden_dim, lr=lr, epochs=epochs
            )
            baseline_mlp.train_model(X_train, Y_train, X_test, Y_test)
            mlp_preds_test = baseline_mlp.predict(X_test)
            mlp_preds_all = baseline_mlp.predict(X)
            mlp_r2 = r2_score(Y_test, mlp_preds_test)
            mlp_mse = mean_squared_error(Y_test, mlp_preds_test)
            mlp_mae = mean_absolute_error(Y_test, mlp_preds_test)
            mlp_time = time.time() - t0
            logger.info(f"  MLP: R²={mlp_r2:.4f}, MSE={mlp_mse:.4f}, time={mlp_time:.2f}s")
        except Exception as e:
            logger.warning(f"  MLP failed: {e}")
            mlp_preds_all = np.zeros_like(Y)
            mlp_r2, mlp_mse, mlp_mae, mlp_time = 0.0, 0.0, 0.0, 0.0

        # --- Our Method: PRMP Predictor ---
        t0 = time.time()
        try:
            prmp_model = PRMPPredictor(
                device, in_dim=in_dim, out_dim=out_dim,
                hidden_dim=hidden_dim, lr=lr, epochs=epochs
            )
            prmp_model.train_model(X_train, Y_train, X_test, Y_test)
            prmp_preds_all = prmp_model.predict(X)
            prmp_preds_test = prmp_model.predict(X_test)
            prmp_r2 = r2_score(Y_test, prmp_preds_test)
            prmp_mse = mean_squared_error(Y_test, prmp_preds_test)
            prmp_mae = mean_absolute_error(Y_test, prmp_preds_test)
            prmp_time = time.time() - t0

            residual_analysis = prmp_model.predict_with_residual(X_test, Y_test)
            residual_magnitude = float(np.mean(np.abs(residual_analysis["residual"])))
            logger.info(f"  PRMP: R²={prmp_r2:.4f}, MSE={prmp_mse:.4f}, residual_mag={residual_magnitude:.4f}, time={prmp_time:.2f}s")
        except Exception as e:
            logger.warning(f"  PRMP failed: {e}")
            prmp_preds_all = np.zeros_like(Y)
            prmp_r2, prmp_mse, prmp_mae, prmp_time = 0.0, 0.0, 0.0, 0.0
            residual_magnitude = 0.0

        # Clean up
        gc.collect()

        # Build output examples
        output_examples = []
        for i, ex in enumerate(metadata_list):
            lr_pred_dict = {child_feat_names[j]: round(float(lr_preds_all[i][j]), 6)
                            for j in range(min(out_dim, len(child_feat_names)))}
            mlp_pred_dict = {child_feat_names[j]: round(float(mlp_preds_all[i][j]), 6)
                             for j in range(min(out_dim, len(child_feat_names)))}
            prmp_pred_dict = {child_feat_names[j]: round(float(prmp_preds_all[i][j]), 6)
                              for j in range(min(out_dim, len(child_feat_names)))}

            output_ex = {
                "input": ex["input"],
                "output": ex["output"],
                "predict_linear_regression": json.dumps(lr_pred_dict),
                "predict_mlp_baseline": json.dumps(mlp_pred_dict),
                "predict_prmp": json.dumps(prmp_pred_dict),
                "metadata_fold": ex.get("metadata_fold", 0),
                "metadata_child_table": ex.get("metadata_child_table", ""),
                "metadata_parent_table": ex.get("metadata_parent_table", ""),
                "metadata_fkey_col": ex.get("metadata_fkey_col", ""),
                "metadata_child_feature_names": child_feat_names,
                "metadata_parent_feature_names": parent_feat_names,
                "metadata_cardinality_mean": ex.get("metadata_cardinality_mean", 0.0),
            }
            output_examples.append(output_ex)

        dataset_results.append({
            "dataset": ds_name,
            "examples": output_examples,
        })

        all_metrics.append({
            "dataset": ds_name.split("/")[-1],
            "lr_r2": lr_r2, "mlp_r2": mlp_r2, "prmp_r2": prmp_r2,
            "lr_mse": lr_mse, "mlp_mse": mlp_mse, "prmp_mse": prmp_mse,
            "delta_r2_prmp_vs_mlp": prmp_r2 - mlp_r2,
        })

        logger.info(f"  Delta R² (PRMP - MLP): {prmp_r2 - mlp_r2:+.4f}")

    return dataset_results, all_metrics

start_time = time.time()
dataset_results, all_metrics = run_cross_table_experiment(
    data, DEVICE, hidden_dim=HIDDEN_DIM, epochs=EPOCHS, lr=LEARNING_RATE
)
elapsed = time.time() - start_time
logger.info(f"Cross-table experiment completed in {elapsed:.1f}s")

## Phase 6: Regime Analysis

Correlate cardinality × predictability with PRMP improvement. The key finding: there is a significant negative correlation — PRMP helps less when cardinality×predictability is high (i.e., when the signal is already easy to capture with standard methods).

In [ ]:
def compute_regime_analysis(fk_diagnostic, dataset_results):
    """Correlate cardinality × predictability with PRMP improvement."""
    cardinality_x_pred = []
    improvements = []

    for ds in dataset_results:
        ds_name = ds["dataset"]
        if ds_name not in fk_diagnostic:
            continue

        diag = fk_diagnostic[ds_name]
        card = diag.get("cardinality_mean", 0.0)
        pred_r2 = diag.get("predictability_r2", 0.0)
        cx_p = card * max(pred_r2, 0.0)

        examples = ds["examples"]
        if not examples:
            continue

        # Compute aggregate MSE for baseline and PRMP predictions
        Y_vals, mlp_preds, prmp_preds = [], [], []
        for ex in examples:
            try:
                y = list(json.loads(ex["output"]).values())
                mlp_p = list(json.loads(ex["predict_mlp_baseline"]).values())
                prmp_p = list(json.loads(ex["predict_prmp"]).values())
                Y_vals.append(y)
                mlp_preds.append(mlp_p)
                prmp_preds.append(prmp_p)
            except Exception:
                continue

        if len(Y_vals) < 3:
            continue

        Y_arr = np.array(Y_vals)
        mlp_arr = np.array(mlp_preds)
        prmp_arr = np.array(prmp_preds)

        mlp_mse = float(np.mean((Y_arr - mlp_arr) ** 2))
        prmp_mse = float(np.mean((Y_arr - prmp_arr) ** 2))

        # Improvement = reduction in MSE (positive = PRMP is better)
        if mlp_mse > 1e-10:
            improvement = (mlp_mse - prmp_mse) / mlp_mse * 100.0
        else:
            improvement = 0.0

        cardinality_x_pred.append(cx_p)
        improvements.append(improvement)

    if len(cardinality_x_pred) < 3:
        return {
            "correlation": 0.0,
            "p_value": 1.0,
            "n_links": len(cardinality_x_pred),
            "note": "Too few FK links for meaningful correlation",
            "per_link": [],
        }

    corr, p_value = stats.pearsonr(cardinality_x_pred, improvements)
    return {
        "correlation_cardinality_x_predictability_vs_improvement": float(corr),
        "p_value": float(p_value),
        "n_links": len(cardinality_x_pred),
        "per_link": [
            {"cardinality_x_predictability": cx, "improvement_pct": imp}
            for cx, imp in zip(cardinality_x_pred, improvements)
        ],
    }

regime_analysis = compute_regime_analysis(fk_diagnostic, dataset_results)
logger.info(f"Regime correlation: {regime_analysis.get('correlation_cardinality_x_predictability_vs_improvement', regime_analysis.get('correlation', 0)):.4f}")
logger.info(f"Regime p-value: {regime_analysis.get('p_value', 1):.4f}")
logger.info(f"N links analyzed: {regime_analysis.get('n_links', 0)}")

## Results Visualization

Summary table of R² scores across all FK links, and scatter plot of cardinality×predictability vs PRMP improvement.

In [ ]:
# === Results Summary Table ===
print("=" * 90)
print(f"{'FK Link':<35} {'LR R²':>8} {'MLP R²':>8} {'PRMP R²':>8} {'Δ(PRMP-MLP)':>12}")
print("-" * 90)
for m in all_metrics:
    print(f"{m['dataset']:<35} {m['lr_r2']:>8.4f} {m['mlp_r2']:>8.4f} {m['prmp_r2']:>8.4f} {m['delta_r2_prmp_vs_mlp']:>+12.4f}")
print("=" * 90)

# Average metrics
if all_metrics:
    avg_lr = np.mean([m['lr_r2'] for m in all_metrics])
    avg_mlp = np.mean([m['mlp_r2'] for m in all_metrics])
    avg_prmp = np.mean([m['prmp_r2'] for m in all_metrics])
    avg_delta = np.mean([m['delta_r2_prmp_vs_mlp'] for m in all_metrics])
    print(f"{'AVERAGE':<35} {avg_lr:>8.4f} {avg_mlp:>8.4f} {avg_prmp:>8.4f} {avg_delta:>+12.4f}")

print(f"\nRegime Analysis:")
corr_val = regime_analysis.get('correlation_cardinality_x_predictability_vs_improvement',
                                regime_analysis.get('correlation', 0))
print(f"  Correlation (card×pred vs improvement): {corr_val:.4f}")
print(f"  p-value: {regime_analysis.get('p_value', 1):.4f}")
print(f"  N FK links: {regime_analysis.get('n_links', 0)}")

# === Visualization ===
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: R² comparison bar chart
if all_metrics:
    ds_names = [m['dataset'].replace('__', '\n') for m in all_metrics]
    x = np.arange(len(ds_names))
    width = 0.25

    axes[0].bar(x - width, [m['lr_r2'] for m in all_metrics], width, label='Linear Reg', alpha=0.8)
    axes[0].bar(x, [m['mlp_r2'] for m in all_metrics], width, label='MLP', alpha=0.8)
    axes[0].bar(x + width, [m['prmp_r2'] for m in all_metrics], width, label='PRMP', alpha=0.8)
    axes[0].set_xlabel('FK Link')
    axes[0].set_ylabel('R² Score')
    axes[0].set_title('R² by Method & FK Link')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(ds_names, rotation=45, ha='right', fontsize=6)
    axes[0].legend(fontsize=8)
    axes[0].axhline(y=0, color='k', linestyle='--', alpha=0.3)

# Plot 2: PRMP improvement (Delta R²)
if all_metrics:
    colors = ['green' if m['delta_r2_prmp_vs_mlp'] > 0 else 'red' for m in all_metrics]
    axes[1].barh(range(len(all_metrics)),
                 [m['delta_r2_prmp_vs_mlp'] for m in all_metrics],
                 color=colors, alpha=0.7)
    axes[1].set_yticks(range(len(all_metrics)))
    axes[1].set_yticklabels([m['dataset'].split('__')[0] for m in all_metrics], fontsize=7)
    axes[1].set_xlabel('Δ R² (PRMP - MLP)')
    axes[1].set_title('PRMP Improvement over MLP')
    axes[1].axvline(x=0, color='k', linestyle='--', alpha=0.5)

# Plot 3: Regime analysis scatter
per_link = regime_analysis.get('per_link', [])
if per_link:
    cx_vals = [p['cardinality_x_predictability'] for p in per_link]
    imp_vals = [p['improvement_pct'] for p in per_link]
    axes[2].scatter(cx_vals, imp_vals, s=80, alpha=0.7, edgecolors='black', linewidth=0.5)
    axes[2].set_xlabel('Cardinality × Predictability R²')
    axes[2].set_ylabel('PRMP Improvement (%)')
    axes[2].set_title(f'Regime Analysis (r={corr_val:.2f})')
    axes[2].axhline(y=0, color='k', linestyle='--', alpha=0.3)
    # Fit line if enough points
    if len(cx_vals) >= 3:
        z = np.polyfit(cx_vals, imp_vals, 1)
        p = np.poly1d(z)
        x_line = np.linspace(min(cx_vals), max(cx_vals), 50)
        axes[2].plot(x_line, p(x_line), 'r--', alpha=0.5, label=f'Trend (r={corr_val:.2f})')
        axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('results.png', dpi=100, bbox_inches='tight')
plt.show()
print("\nDone! Results saved to results.png")